# 🧠 DyslexiaLens — Data Science Notebook
## Persiapan Dataset & Preprocessing

**Tema Capstone:** Accessible & Adaptive Learning  
**Peran:** Data Scientist  
**Dataset:** Gambo — Dyslexia Handwriting Dataset (Kaggle)

---

### Deskripsi Singkat
Notebook ini mencakup seluruh proses persiapan data (*Data Wrangling*) untuk proyek **DyslexiaLens**, sebuah sistem *early screening* disleksia berbasis analisis citra tulisan tangan.

Sistem ini **bukan alat diagnosis medis**, melainkan alat bantu skrining awal yang memberikan indikasi dini kepada orang tua dan pendidik sebelum pemeriksaan lebih lanjut oleh profesional.

---

### Alur Notebook
| Tahap | Deskripsi | Sifat |
|---|---|---|
| **Tahap 1** | Assessing Data — Evaluasi kualitas & struktur dataset, audit distribusi dan anomali | Wajib |
| **Tahap 2** | Physical Renaming — Normalisasi skala nama file fisik ke skala 1–6 | Opsional |
| **Tahap 3** | Cleaning Data & CSV Generation — Filter data kotor, buat master CSV | Wajib |
| **Tahap 4** | EDA & Visualisasi Data — Analisis distribusi, imbalance, dan pola visual | Wajib |
| **Tahap 5** | Data Augmentation — Menyeimbangkan distribusi severity score via rotasi & scaling | Wajib |

## 📌 Pertanyaan Bisnis

Notebook ini dirancang untuk menjawab dua pertanyaan bisnis utama yang terukur:

**1. Dapatkah pola goresan tulisan tangan digunakan sebagai indikator awal disleksia?**
> Hipotesis: Tulisan dari penderita disleksia memiliki pola goresan yang secara visual dapat dibedakan dari tulisan normal — baik melalui pembalikan arah huruf (*Reversal*) maupun koreksi berulang yang meninggalkan bekas coretan (*Corrected*).

**2. Seberapa parah tingkat gejala disleksia berdasarkan keparahan coretan (Severity Score 0–6)?**
> Hipotesis: Tingkat keparahan goresan dapat dikuantifikasi secara bertingkat, di mana skor lebih tinggi merepresentasikan distorsi tulisan yang lebih ekstrem dan lebih kuat mengindikasikan gejala disleksia.

---

### Definisi Variabel Target

| Variabel | Tipe | Nilai | Keterangan |
|---|---|---|---|
| `target_class` | Integer | `0` = Normal, `1` = Disleksia | Label biner untuk klasifikasi utama |
| `severity_score` | Integer | `0` = Sehat, `1`–`6` = Ringan hingga Parah | Skor keparahan gejala disleksia |
| `folder_category` | String | `Normal`, `Corrected`, `Reversal` | Kategori kelas asal dari dataset |

> **⚠️ Catatan Anti Data Leakage:** Variabel `severity_score` dan `folder_category` **tidak boleh dimasukkan sebagai fitur input model** karena merupakan turunan langsung dari label. Hanya `image_path` yang menjadi input — dan `target_class` yang menjadi output target utama.

## 📦 Sumber Dataset & Referensi Ilmiah

### Identitas Dataset
| Atribut | Detail |
|---|---|
| **Nama** | Dyslexia Handwriting Dataset (Gambo) |
| **Sumber** | Kaggle — Dataset Publik |
| **Link** | https://www.kaggle.com/datasets/drizasazanitaisa/dyslexia-handwriting-dataset |
| **Total Gambar** | 208.372 file `.png` |
| **Format** | Grayscale, resolusi 28×28 piksel |
| **Kelas** | `Normal`, `Corrected`, `Reversal` |

### Cara Mendapatkan Dataset
Dataset tidak disertakan di dalam repositori Git (di-*gitignore*) karena ukurannya yang besar.  
Untuk menjalankan notebook ini secara **lokal**, lakukan langkah berikut:
1. Unduh dataset dari link Kaggle di atas.
2. Ekstrak dan letakkan folder `Gambo/` di direktori yang sama dengan file notebook ini.
3. Pastikan struktur folder: `Gambo/Train/` dan `Gambo/Test/` sudah tersedia.

Untuk menjalankan di **Google Colab**:
1. Upload folder `Gambo/` ke Google Drive Anda.
2. Mount Google Drive di Colab, lalu ubah variabel `root_dir` di setiap cell menjadi path Drive Anda (misal: `'/content/drive/MyDrive/Gambo'`).

### Referensi Ilmiah
Dataset ini dipublikasikan dan digunakan dalam penelitian berikut:

1. M. S. A. B. Rosli, I. S. Isa, S. A. Ramlan, S. N. Sulaiman and M. I. F. Maruzuki, *"Development of CNN Transfer Learning for Dyslexia Handwriting Recognition"*, 2021 11th IEEE International Conference on Control System, Computing and Engineering (ICCSCE), 2021, pp. 194-199, doi: 10.1109/ICCSCE52189.2021.9530971.

2. N. S. L. Seman, I. S. Isa, S. A. Ramlan, W. Li-Chih and M. I. F. Maruzuki, *"Classification of Handwriting Impairment Using CNN for Potential Dyslexia Symptom"*, 2021 11th IEEE International Conference on Control System, Computing and Engineering (ICCSCE), 2021, pp. 188-193, doi: 10.1109/ICCSCE52189.2021.9530989.

3. Isa, Iza Sazanita. *CNN Comparisons Models On Dyslexia Handwriting Classification*. Universiti Teknologi MARA Cawangan Pulau Pinang, 2021.

4. Isa, I. S., Rahimi, W. N. S., Ramlan, S. A., & Sulaiman, S. N. (2019). *Automated detection of dyslexia symptom based on handwriting image for primary school children*. Procedia Computer Science, 163, 440-449.

---
# Tahap 1: Assessing Data — Evaluasi Kualitas & Struktur Dataset

Sebelum melakukan pembersihan data, kita perlu memahami **apa yang ada di dalam dataset** secara menyeluruh. Tahap *Assessing Data* ini meliputi:
1. Mengecek distribusi jumlah file per kelas dan per split
2. Memvalidasi konsistensi format (ekstensi, resolusi, mode warna)
3. Mengaudit penamaan file untuk menemukan anomali label

---

### Temuan Utama (Hasil Audit Manual + Algoritmik)

**✅ Temuan 1 — Sistem Severity Score (Kunci)**  
Folder numerik `1, 4, 5, 6, 7, 8, 9` di dalam kelas `Corrected` dan `Reversal` **bukan** merepresentasikan karakter angka yang ditulis, melainkan **Tingkat Keparahan Goresan (Severity Score)**. Inspeksi visual membuktikan hampir seluruh abjad tersebar merata di tiap folder — tidak ada konsentrasi karakter tertentu. Ini berarti dataset **tidak mengalami Data Leakage** dan model akan dipaksa belajar pola goresan, bukan menghafal karakter.

| Skor Asli (Periset) | Kondisi Visual | Keparahan Nyata | Skor AI (Setelah Normalisasi) |
|---|---|---|---|
| `1` | Goresan hancur lebur, hampir tidak terbaca | **Paling Parah** | `6` |
| `4` | Banyak timpa-menimpa, kerangka huruf samar | Parah | `6` (digabung) |
| `5`–`8` | Koreksi tampak, huruf masih bisa ditebak | Menengah | `5` → `2` |
| `9` | Goresan ringan, huruf aslinya mudah dikenali | **Paling Ringan** | `1` |

> ⚠️ Skala asli bersifat **terbalik** — `1` = Parah (bukan Ringan), `9` = Ringan (bukan Parah). Jika langsung digunakan sebagai label angka, model AI akan menganggap tulisan paling parah (skor 1) justru paling dekat dengan Normal (skor 0). Ini dikoreksi melalui *Dictionary Mapping* di Tahap 2 & Tahap 3.

---

**⚠️ Temuan 2 — Kontaminasi Label (Label Noise) di Kelas Non-Normal**  
Ditemukan file bernama `NormalXXXX.png` yang terselip di dalam folder `Corrected` dan `Reversal`. Secara visual, konten gambar-gambar tersebut adalah goresan cacat — bukan tulisan tangan normal yang bersih. Ini adalah *Label Noise* yang terjadi kemungkinan akibat *script rename otomatis* tanpa verifikasi visual dari pihak pembuat dataset asli.

---

**⚠️ Temuan 3 — Anomali Visual di Kelas Normal Asli**  
Melalui inspeksi visual manual pada sampel gambar di dalam kelas `Normal`, ditemukan bahwa **tidak semua gambar di kelas Normal benar-benar bersih**. Sejumlah sampel menampilkan goresan koreksi atau pola yang secara visual menyerupai karakteristik `Corrected`, meskipun secara nama file dan letak folder dikategorikan sebagai `Normal`.

Ini memperkuat indikasi adanya **Kontaminasi Label Dua Arah** dari dataset asli:
- File bergoresan cacat masuk ke folder Normal (`NormalXXXX.png` tersesat)
- File bertulisan normal yang secara visual wajar, namun labelnya terkontaminasi oleh goresan lingkungan sekitar saat pengambilan data

> 📌 Temuan ini menjadi dasar keputusan untuk **tidak semata-mata mengandalkan nama folder** sebagai label tunggal, dan mendukung penggunaan `master_dataset_dyslexia.csv` dengan filter eksplisit sebagai sumber kebenaran (*ground truth*) untuk proses training.

### 1A. Statistik Distribusi Dataset

In [ ]:
import os
from pathlib import Path
from collections import defaultdict

root_dir = r'Gambo'

stats = defaultdict(lambda: defaultdict(int))
total = 0
noise_count = 0
non_png_count = 0

for root, dirs, files in os.walk(root_dir):
    for file in files:
        ext = os.path.splitext(file)[1].lower()
        if ext != '.png':
            non_png_count += 1
            continue
        parts = Path(root).parts
        try:
            split_type = parts[-2]
            category  = parts[-1]
        except:
            continue
        stats[split_type][category] += 1
        total += 1
        if 'Normal' in file and category != 'Normal':
            noise_count += 1

print('=' * 55)
print('     STATISTIK DISTRIBUSI DATASET GAMBO')
print('=' * 55)
for split, categories in sorted(stats.items()):
    subtotal = sum(categories.values())
    print(f'\n📁 {split} ({subtotal:,} gambar)')
    for cat, count in sorted(categories.items()):
        print(f'   └── {cat:12s}: {count:>7,} gambar')

print(f'\n{"-" * 55}')
print(f'  Total file .png          : {total:,}')
print(f'  File non-.png ditemukan  : {non_png_count:,}  (diabaikan)')
print(f'  File Noise terdeteksi    : {noise_count:,}  (akan di-drop di Tahap 3)')
print(f'  Estimasi data bersih     : {total - noise_count:,}')
print('=' * 55)

### 1B. Verifikasi Integritas File, Resolusi & Format Warna

Kode berikut melakukan *sampling* acak dari setiap kelas untuk memverifikasi:
- Apakah file dapat dibuka (tidak corrupt)
- Konsistensi resolusi (ukuran piksel)
- Konsistensi format warna (*color mode*)

In [ ]:
import os
import random
from pathlib import Path
from collections import Counter, defaultdict
from PIL import Image

root_dir = r'Gambo'
SAMPLE_PER_CLASS = 500

class_files = defaultdict(list)
for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.endswith('.png'):
            parts = Path(root).parts
            try:
                category = parts[-1]
            except:
                continue
            class_files[category].append(os.path.join(root, file))

print('=' * 55)
print('  VERIFIKASI INTEGRITAS, RESOLUSI & FORMAT WARNA')
print('=' * 55)

for category, paths in sorted(class_files.items()):
    sample = random.sample(paths, min(SAMPLE_PER_CLASS, len(paths)))
    resolutions = Counter()
    modes = Counter()
    corrupt = 0
    for path in sample:
        try:
            with Image.open(path) as img:
                resolutions[img.size] += 1
                modes[img.mode] += 1
        except Exception:
            corrupt += 1

    print(f'\n📂 Kelas: {category} (sample {len(sample)} dari {len(paths):,} file)')
    print(f'   File corrupt        : {corrupt}')
    print(f'   Resolusi ditemukan  : {dict(resolutions.most_common(3))}')
    print(f'   Format warna        : {dict(modes)}')

print('\n' + '=' * 55)
print('  Verifikasi selesai.')
print('=' * 55)

### Kesimpulan Assessing

| Aspek | Status | Keterangan |
|---|---|---|
| Struktur folder | ✅ | `Train/Test → Normal/Corrected/Reversal` — hierarki bersih |
| Format file | ✅ | 100% berformat `.png`, nol file non-PNG |
| Integritas file | ✅ | Nol file corrupt dari hasil sampling |
| Resolusi gambar | ✅ | 28×28 piksel (identik dengan format MNIST/EMNIST) |
| Format warna | ✅ | Grayscale (`L`) atau Binary (`1`) — tidak ada RGB |
| Distribusi kelas | ⚠️ | Corrected lebih dominan — perlu pantau *class imbalance* |
| Severity Score | ✅ | Folder 1–9 = derajat keparahan (bukan karakter), skala terbalik |
| Label Noise (kelas non-Normal) | ⚠️ | File `NormalXXXX.png` terselip — di-*drop* di Tahap 3 |
| Anomali visual kelas Normal | ⚠️ | Sebagian sampel Normal mengandung goresan tidak wajar — tercatat sebagai risiko residual |

---
# Tahap 2: Preprocessing Tambahan (Physical Renaming) — Opsional
Kode opsional ini secara fisik mengganti nama file di local disk Anda agar skala keparahannya berurutan menjadi skala **1 sampai 6** (1 = Paling Ringan, 6 = Paling Parah) menggunakan *Dictionary Mapping*:

| Skor Asli | → | Skor Baru | Keterangan |
|---|---|---|---|
| `9` | → | `1` | Paling Ringan |
| `8` | → | `2` | |
| `7` | → | `3` | |
| `6` | → | `4` | |
| `5` | → | `5` | |
| `4` | → | `6` | Corrected Paling Parah |
| `1` | → | `6` | Reversal (digabung ke puncak) |

> ⚠️ **Jalankan sel ini hanya SEKALI.** Menjalankannya dua kali pada dataset yang sudah di-rename akan menyebabkan skala bergeser dan data menjadi kacau.
>
> ℹ️ **Opsional:** Jika Anda melewati Tahap 2, kode Tahap 3 tetap berfungsi karena `get_score` sudah dirancang untuk menangani **kedua kondisi** (skala asli dan skala setelah di-rename).

In [11]:
import os
import time

root_dir = r'Gambo'

print("Mempersiapkan penggantian nama (Physical Renaming) skala 1-6...")

score_map = {
    9: 1, 8: 2, 7: 3, 6: 4, 5: 5,
    4: 6, # Asli 4 (Corrected Paling Parah) -> Jadi Skor AI 6
    1: 6  # Asli 1 (Reversal) -> Sama-sama disatukan di puncak Skor AI 6
}

to_rename = []
for root, dirs, files in os.walk(root_dir):
    if not ('Corrected' in root or 'Reversal' in root):
        continue
    for file in files:
        if file.endswith('.png'):
            prefix, separator = None, None
            if '_' in file:
                head = file.split('_')[0]
                if head.isdigit() and int(head) in score_map:
                    prefix, separator = int(head), '_'
            elif '-' in file:
                head = file.split('-')[0]
                if head.isdigit() and int(head) in score_map:
                    prefix, separator = int(head), '-'
            if prefix is not None:
                new_prefix = score_map[prefix]
                sisa_nama = file.split(separator, 1)[1]
                new_name = f"{new_prefix}{separator}{sisa_nama}"
                to_rename.append((os.path.join(root, file), os.path.join(root, new_name)))

print(f"Total file yang akan dimapping ulang namanya: {len(to_rename)} file.")

temp_rename = []
for old, new in to_rename:
    temp_path = new + ".TEMP"
    os.rename(old, temp_path)
    temp_rename.append((temp_path, new))

for temp, final_new in temp_rename:
    berhasil = False
    for _ in range(10):
        try:
            os.replace(temp, final_new)
            berhasil = True
            break
        except PermissionError:
            time.sleep(0.1)
    if not berhasil:
        print(f"Gagal: {final_new}")

print("Selesai! Seluruh nama fisik file berhasil dipetakan ke skala konsisten (1 Ringan -> 6 Parah)!")

Mempersiapkan penggantian nama (Physical Renaming) skala 1-6...
Total file yang akan dimapping ulang namanya: 121835 file.
Selesai! Seluruh nama fisik file berhasil dipetakan ke skala konsisten (1 Ringan -> 6 Parah)!


---
# Tahap 3: Cleaning Data & Pembuatan Master CSV

Kode ini melakukan **Logical Cleaning** — memfilter semua file kotor (*Label Noise*) langsung dari tabel Pandas **tanpa menghapus file fisik aslinya**, lalu menyimpan daftar data bersih ke `master_dataset_dyslexia.csv`.

Fungsi `get_score` dirancang untuk bekerja dalam **dua kondisi secara otomatis**:
- **Jika Tahap 2 dijalankan:** nama file sudah berprefix 1–6, kode membaca langsung.
- **Jika Tahap 2 dilewati:** nama file masih berprefix asli (1, 4, 5, 6, 7, 8, 9), kode menerapkan *Dictionary Mapping* secara otomatis ke rentang 0–6.

> **INGAT:** Ubah variabel `root_dir` sesuai lokasi dataset Anda (lokal atau Google Drive/Colab).

In [12]:
import os
import pandas as pd
from pathlib import Path

root_dir = r'Gambo'

# Dictionary Mapping: Skor asli periset (terbalik) -> Skor AI (linear 0-6)
# Digunakan sebagai fallback jika Tahap 2 (Physical Renaming) dilewati.
SCORE_MAP_ORIGINAL = {1: 6, 4: 6, 5: 5, 6: 4, 7: 3, 8: 2, 9: 1}

print("Membaca seluruh direktori...")
data = []

for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.endswith('.png'):
            parts = Path(root).parts
            try:
                split_type = parts[-2]
                category = parts[-1]
            except:
                continue
            path_full = os.path.join(root, file)
            data.append({
                'image_path': path_full,
                'file_name': file,
                'split': split_type,
                'folder_category': category
            })

df = pd.DataFrame(data)
print(f"Total gambar berserakan yang ditemukan: {len(df)} file.\n")

def get_score(row):
    """Ekstrak & normalisasi Severity Score dari nama file.
    Bekerja otomatis untuk dataset pre-Tahap 2 (skor asli) maupun post-Tahap 2 (skor 1-6).
    """
    filename = row['file_name']
    folder = row['folder_category']

    # Buang file NormalXXXX yang tersesat ke kelas non-Normal (Label Noise)
    if 'Normal' in filename and folder != 'Normal':
        return 'DROP'

    # Kelas Normal -> Skor 0 (Sehat)
    if folder == 'Normal':
        return 0

    # Kelas Corrected & Reversal -> Ekstrak prefix angka
    if folder in ['Corrected', 'Reversal']:
        for sep in ['_', '-']:
            if sep in filename:
                prefix = filename.split(sep)[0]
                if prefix.isdigit():
                    val = int(prefix)
                    if 1 <= val <= 6:
                        # Post-Tahap 2: nama file sudah di-rename ke skala 1-6
                        return val
                    elif val in SCORE_MAP_ORIGINAL:
                        # Pre-Tahap 2: terapkan Dictionary Mapping ke skala AI 0-6
                        return SCORE_MAP_ORIGINAL[val]
                break

    return 'DROP'  # Buang file tak dikenal

df['severity_score'] = df.apply(get_score, axis=1)
df_clean = df[~df['severity_score'].astype(str).str.contains('DROP')].copy()
df_clean['severity_score'] = df_clean['severity_score'].astype(int)
df_clean['target_class'] = df_clean['severity_score'].apply(lambda x: 0 if x == 0 else 1)

output_csv = 'master_dataset_dyslexia.csv'
df_clean.to_csv(output_csv, index=False)

print(f"SUCCESS! Disimpan ke '{output_csv}' sebanyak {len(df_clean)} file gambar bersih.")
df_clean.sample(5)

Membaca seluruh direktori...
Total gambar berserakan yang ditemukan: 208372 file.

SUCCESS! Disimpan ke 'master_dataset_dyslexia.csv' sebanyak 180726 file gambar bersih.


---
# Tahap 4: Exploratory Data Analysis (EDA)

Tahap ini bertujuan untuk **menggali insight visual** dari dataset yang sudah dibersihkan di Tahap 3. EDA akan menjawab pertanyaan-pertanyaan berikut:
1. Bagaimana distribusi jumlah gambar antar kelas dan antar split?
2. Apakah ada *class imbalance* yang perlu diwaspadai?
3. Bagaimana distribusi Severity Score (0–6)?
4. Seperti apa perbedaan visual antara kelas Normal, Corrected, dan Reversal?
5. Seperti apa perbedaan visual antara Severity Score rendah (1) dan tinggi (6)?

> **Catatan:** Semua visualisasi di bawah ini menggunakan data dari `master_dataset_dyslexia.csv` yang dihasilkan di Tahap 3.

### 4A. Load Data Bersih dari CSV

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import numpy as np
import os

# Konfigurasi visual
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# Load CSV hasil Tahap 3
df = pd.read_csv('master_dataset_dyslexia.csv')
print(f'Total data bersih: {len(df):,} baris')
print(f'Kolom: {list(df.columns)}')
print(f'\nDistribusi target_class:')
print(df['target_class'].value_counts())
df.head()

### 4B. Distribusi Kelas per Split (Train vs Test)

Visualisasi ini menjawab: **Apakah proporsi kelas (`Normal`, `Corrected`, `Reversal`) konsisten antara set Train dan Test?**
Jika tidak konsisten, model bisa mengalami *distribution shift* saat evaluasi.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, split in enumerate(['Train', 'Test']):
    data = df[df['split'] == split]['folder_category'].value_counts()
    colors = ['#2ecc71', '#e74c3c', '#3498db']
    bars = axes[i].bar(data.index, data.values, color=colors, edgecolor='white', linewidth=0.8)
    axes[i].set_title(f'Distribusi Kelas — {split}', fontweight='bold', fontsize=13)
    axes[i].set_ylabel('Jumlah Gambar')
    axes[i].set_xlabel('Kelas')
    for bar in bars:
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
                     f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Distribusi Kelas Dataset per Split', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Insight 4B:**
- Kelas `Corrected` mendominasi baik di Train maupun Test — ini mengindikasikan potensi *class imbalance*.
- Kelas `Normal` paling sedikit jumlahnya, yang bisa menyebabkan model bias terhadap kelas mayoritas.
- Proporsi relatif antar kelas cukup konsisten antara Train dan Test (tidak ada *distribution shift* yang ekstrem).

### 4C. Distribusi Severity Score (0–6)

Visualisasi ini menampilkan frekuensi tiap level keparahan. Skor `0` = Normal, skor `1`–`6` = Disleksia dengan derajat meningkat.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

score_counts = df['severity_score'].value_counts().sort_index()
colors_severity = ['#2ecc71'] + sns.color_palette('YlOrRd', 6).as_hex()

bars = ax.bar(score_counts.index, score_counts.values, color=colors_severity,
              edgecolor='white', linewidth=0.8)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xlabel('Severity Score', fontsize=12)
ax.set_ylabel('Jumlah Gambar', fontsize=12)
ax.set_title('Distribusi Severity Score (0 = Normal, 6 = Paling Parah)', fontweight='bold', fontsize=13)
ax.set_xticks(range(7))
ax.set_xticklabels(['0\n(Normal)', '1\n(Ringan)', '2', '3', '4', '5', '6\n(Parah)'])

plt.tight_layout()
plt.show()

**Insight 4C:**
- Distribusi Severity Score **tidak merata** — terdapat konsentrasi tertentu pada beberapa level.
- Skor `0` (Normal) memiliki jumlah yang signifikan sebagai *baseline*.
- Skor `6` (paling parah, gabungan Reversal + Corrected terparah) kemungkinan jumlahnya paling besar karena menggabungkan dua skor asli (`1` dan `4`).
- *Class imbalance* pada level severity ini perlu ditangani saat training (misalnya via *class weights* atau *stratified sampling*).

### 4D. Analisis Class Imbalance — Normal vs Disleksia (Binary)

Karena tugas utama model adalah **klasifikasi biner** (`target_class`: 0 vs 1), kita perlu melihat rasio antara kedua kelas ini.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
target_counts = df['target_class'].value_counts().sort_index()
labels = ['Normal (0)', 'Disleksia (1)']
colors_binary = ['#2ecc71', '#e74c3c']
axes[0].pie(target_counts, labels=labels, colors=colors_binary, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'},
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Proporsi Binary Class', fontweight='bold', fontsize=13)

# Bar chart per split
binary_split = df.groupby(['split', 'target_class']).size().unstack(fill_value=0)
binary_split.columns = labels
binary_split.plot(kind='bar', ax=axes[1], color=colors_binary, edgecolor='white', linewidth=0.8)
axes[1].set_title('Binary Class per Split', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Jumlah Gambar')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(loc='upper right')

for container in axes[1].containers:
    axes[1].bar_label(container, fmt='{:,.0f}', fontsize=9, fontweight='bold', padding=3)

plt.suptitle('Analisis Class Imbalance', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

ratio = target_counts[1] / target_counts[0]
print(f'Rasio Disleksia : Normal = {ratio:.2f} : 1')
print(f'Normal: {target_counts[0]:,} | Disleksia: {target_counts[1]:,}')

**Insight 4D:**
- Dataset bersifat **imbalanced** — kelas Disleksia jauh lebih banyak dari Normal.
- Rasio yang tidak seimbang ini berpotensi menyebabkan model bias ke arah prediksi Disleksia.
- **Rekomendasi mitigasi untuk AI Engineer:**
  - Gunakan `class_weight='balanced'` atau hitung manual bobot per kelas.
  - Pertimbangkan *stratified split* untuk memastikan kedua kelas terwakili di setiap batch.
  - Evaluasi model dengan **F1-Score** dan **Recall**, bukan hanya Accuracy.

### 4E. Sampel Visual Gambar per Kelas

Menampilkan 5 sampel acak dari setiap kelas untuk memberikan gambaran visual bagaimana perbedaan pola goresan antar kelas.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(12, 8))

categories = ['Normal', 'Corrected', 'Reversal']
for row, cat in enumerate(categories):
    samples = df[df['folder_category'] == cat].sample(5, random_state=42)
    for col, (_, sample) in enumerate(samples.iterrows()):
        img_path = sample['image_path']
        if os.path.exists(img_path):
            img = mpimg.imread(img_path)
            axes[row][col].imshow(img, cmap='gray')
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_ylabel(cat, fontsize=13, fontweight='bold', rotation=0, labelpad=60)

plt.suptitle('Sampel Gambar per Kelas (5 Acak per Baris)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Insight 4E:**
- **Normal:** Huruf terbentuk dengan jelas, satu goresan dominan, minim noise. Bentuk karakter mudah dikenali.
- **Corrected:** Tampak goresan tumpang-tindih akibat koreksi berulang. Pola ini menghasilkan *scribbling* yang menjadi ciri khas utama disleksia tipe koreksi.
- **Reversal:** Huruf tertulis terbalik (*mirror image*) — misalnya `b` terlihat seperti `d`, atau huruf menghadap arah berlawanan. Ini gejala klasik disleksia.

### 4F. Sampel Visual per Severity Score (1 vs 3 vs 6)

Perbandingan visual antara tingkat keparahan **rendah (1)**, **menengah (3)**, dan **tinggi (6)** untuk melihat apakah perbedaan derajat keparahan dapat dibedakan secara visual.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(12, 8))

severity_levels = [2, 4, 6]
severity_labels = ['Skor 2 (Paling Ringan)', 'Skor 4 (Menengah)', 'Skor 6 (Parah)']

for row, (score, label) in enumerate(zip(severity_levels, severity_labels)):
    samples = df[df['severity_score'] == score].sample(5, random_state=42)
    for col, (_, sample) in enumerate(samples.iterrows()):
        img_path = sample['image_path']
        if os.path.exists(img_path):
            img = mpimg.imread(img_path)
            axes[row][col].imshow(img, cmap='gray')
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_ylabel(label, fontsize=11, fontweight='bold', rotation=0, labelpad=80)

plt.suptitle('Perbandingan Visual Severity Score (Rendah → Tinggi)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Insight 4F:**
- **Skor 1 (Ringan):** Goresan koreksi minimal — huruf asli masih sangat mudah dikenali. Distorsi hampir tidak terlihat.
- **Skor 3 (Menengah):** Koreksi mulai tampak — ada goresan tambahan yang menutupi sebagian bentuk huruf asli.
- **Skor 6 (Parah):** Goresan sangat destruktif — huruf hampir tidak bisa ditebak. Ini adalah level di mana *scribbling* sudah dominan.

Perbedaan visual yang jelas antara skor rendah dan tinggi ini **mendukung hipotesis** bahwa Severity Score dapat menjadi variabel target yang bermakna untuk model AI.

### 4G. Analisis Pola per Kategori: Corrected vs Reversal

Membandingkan distribusi severity score **di dalam** masing-masing kelas disleksia untuk memahami karakteristik goresan yang berbeda.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, cat in enumerate(['Corrected', 'Reversal']):
    data = df[df['folder_category'] == cat]['severity_score'].value_counts().sort_index()
    color = '#e74c3c' if cat == 'Corrected' else '#3498db'
    bars = axes[i].bar(data.index, data.values, color=color, edgecolor='white', linewidth=0.8, alpha=0.85)
    axes[i].set_title(f'Distribusi Severity Score — {cat}', fontweight='bold', fontsize=13)
    axes[i].set_xlabel('Severity Score')
    axes[i].set_ylabel('Jumlah Gambar')
    axes[i].set_xticks(range(1, 7))
    for bar in bars:
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                     f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Pola Severity Score per Kategori Disleksia', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Insight 4G:**
- **Corrected:** Memiliki distribusi severity score yang lebih tersebar (skor 1–6), mencerminkan variasi tingkat koreksi tulisan dari ringan hingga ekstrem.
- **Reversal:** Kebanyakan terkonsentrasi di skor `6` (karena skor asli `1` dari periset, yang merupakan reversal paling destruktif, dipetakan ke skor AI tertinggi).
- Perbedaan distribusi ini menunjukkan bahwa **Corrected** dan **Reversal** memiliki karakteristik goresan yang berbeda — model bisa dilatih untuk membedakan keduanya, bukan hanya Normal vs Disleksia.

### Kesimpulan EDA

| Pertanyaan | Temuan |
|---|---|
| Distribusi kelas konsisten antar split? | ✅ Ya — proporsi relatif Train dan Test serupa |
| Ada *class imbalance*? | ⚠️ Ya — Disleksia >> Normal. Perlu `class_weight` saat training |
| Severity Score terdistribusi merata? | ⚠️ Tidak — skor `6` dominan karena penggabungan dua skor asli |
| Perbedaan visual antar kelas jelas? | ✅ Ya — Normal bersih, Corrected bertumpuk, Reversal terbalik |
| Perbedaan visual antar severity jelas? | ✅ Ya — skor rendah masih terbaca, skor tinggi destruktif |
| Pola Corrected vs Reversal berbeda? | ✅ Ya — distribusi severity dan jenis distorsi berbeda |

> **Kesimpulan Utama:** Dataset ini layak digunakan untuk klasifikasi biner (Normal vs Disleksia) maupun multi-class severity. Namun, *severity imbalance* pada skor 2–5 harus dimitigasi — dilakukan di **Tahap 5** berikut.

---
## Visualisasi Explanatory — Menjawab Pertanyaan Bisnis

Bagian ini secara eksplisit menjawab **dua pertanyaan bisnis utama** yang didefinisikan di awal notebook, menggunakan visualisasi dan analisis statistik sebagai bukti pendukung.

> **Berbeda dengan EDA di atas** (yang bersifat *exploratory* / mencari pola), bagian ini bersifat **explanatory** — menyajikan jawaban yang jelas dan terstruktur.

### Pertanyaan Bisnis 1: *"Apakah perbedaan distribusi visual antara kelas Normal dan Disleksia signifikan?"*

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import os
from PIL import Image

# --- Panel 1: Perbandingan visual side-by-side Normal vs Disleksia ---
fig, axes = plt.subplots(2, 6, figsize=(15, 5))

# Baris 1: Normal
normal_samples = df[df['target_class'] == 0].sample(6, random_state=42)
for col, (_, row) in enumerate(normal_samples.iterrows()):
    if os.path.exists(row['image_path']):
        axes[0][col].imshow(mpimg.imread(row['image_path']), cmap='gray')
    axes[0][col].axis('off')
    if col == 0:
        axes[0][col].set_ylabel('Normal', fontsize=13, fontweight='bold', rotation=0, labelpad=55)

# Baris 2: Disleksia (campuran Corrected & Reversal)
dys_samples = df[df['target_class'] == 1].sample(6, random_state=42)
for col, (_, row) in enumerate(dys_samples.iterrows()):
    if os.path.exists(row['image_path']):
        axes[1][col].imshow(mpimg.imread(row['image_path']), cmap='gray')
    axes[1][col].axis('off')
    if col == 0:
        axes[1][col].set_ylabel('Disleksia', fontsize=13, fontweight='bold', rotation=0, labelpad=55)

plt.suptitle('Pertanyaan Bisnis 1: Normal vs Disleksia — Apakah Berbeda Secara Visual?',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --- Panel 2: Rata-rata intensitas piksel per kelas ---
def compute_mean_image(subset, n=200):
    """Hitung rata-rata piksel dari n sampel acak."""
    samples = subset.sample(min(n, len(subset)), random_state=42)
    arrays = []
    for _, row in samples.iterrows():
        try:
            img = Image.open(row['image_path']).convert('L').resize((28, 28))
            arrays.append(np.array(img, dtype=np.float32))
        except:
            continue
    return np.mean(arrays, axis=0)

mean_normal = compute_mean_image(df[df['target_class'] == 0])
mean_dys = compute_mean_image(df[df['target_class'] == 1])

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(mean_normal, cmap='gray')
axes[0].set_title('Rata-rata Normal', fontweight='bold')
axes[0].axis('off')
axes[1].imshow(mean_dys, cmap='gray')
axes[1].set_title('Rata-rata Disleksia', fontweight='bold')
axes[1].axis('off')
axes[2].imshow(np.abs(mean_normal - mean_dys), cmap='hot')
axes[2].set_title('Perbedaan (|Normal - Disleksia|)', fontweight='bold')
axes[2].axis('off')

plt.suptitle('Heatmap Rata-rata Piksel: Normal vs Disleksia', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

diff = np.mean(np.abs(mean_normal - mean_dys))
print(f'Rata-rata selisih intensitas piksel: {diff:.2f} (dari skala 0-255)')
print(f'Semakin tinggi nilainya, semakin jelas perbedaan distribusi visual.')

**Jawaban Pertanyaan Bisnis 1:**

> ✅ **Ya, perbedaan distribusi visual antara kelas Normal dan Disleksia signifikan.**

Bukti pendukung:
1. **Sampel visual** menunjukkan pola goresan yang jelas berbeda — Normal memiliki garis bersih dan tegas, sementara Disleksia menampilkan goresan tumpang-tindih (*scribbling*) atau pembalikan arah huruf.
2. **Heatmap rata-rata piksel** memperlihatkan bahwa distribusi kegelapan antar kelas berbeda secara terukur — area yang menyala di *heatmap* perbedaan menunjukkan zona di mana pola goresan disleksia paling menyimpang dari Normal.
3. Temuan ini **mendukung hipotesis awal**: *"Tulisan dari penderita disleksia memiliki pola goresan yang secara visual dapat dibedakan dari tulisan normal."*

### Pertanyaan Bisnis 2: *"Apakah Severity Score 6 secara visual jauh berbeda dari Severity Score 1?"*

In [ ]:
# --- Perbandingan visual langsung: Skor 2 vs Skor 6 ---
fig, axes = plt.subplots(2, 6, figsize=(15, 5))

# Baris 1: Skor 2 (Paling Ringan)
score1_samples = df[df['severity_score'] == 2].sample(6, random_state=42)
for col, (_, row) in enumerate(score1_samples.iterrows()):
    if os.path.exists(row['image_path']):
        axes[0][col].imshow(mpimg.imread(row['image_path']), cmap='gray')
    axes[0][col].axis('off')
    if col == 0:
        axes[0][col].set_ylabel('Skor 2\n(Ringan)', fontsize=12, fontweight='bold', rotation=0, labelpad=60)

# Baris 2: Skor 6 (Paling Parah)
score6_samples = df[df['severity_score'] == 6].sample(6, random_state=42)
for col, (_, row) in enumerate(score6_samples.iterrows()):
    if os.path.exists(row['image_path']):
        axes[1][col].imshow(mpimg.imread(row['image_path']), cmap='gray')
    axes[1][col].axis('off')
    if col == 0:
        axes[1][col].set_ylabel('Skor 6\n(Parah)', fontsize=12, fontweight='bold', rotation=0, labelpad=60)

plt.suptitle('Pertanyaan Bisnis 2: Severity Score 2 vs 6 — Apakah Berbeda?',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --- Heatmap rata-rata: Skor 2 vs Skor 6 ---
mean_s1 = compute_mean_image(df[df['severity_score'] == 2])
mean_s6 = compute_mean_image(df[df['severity_score'] == 6])

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(mean_s1, cmap='gray')
axes[0].set_title('Rata-rata Skor 2 (Ringan)', fontweight='bold')
axes[0].axis('off')
axes[1].imshow(mean_s6, cmap='gray')
axes[1].set_title('Rata-rata Skor 6 (Parah)', fontweight='bold')
axes[1].axis('off')
axes[2].imshow(np.abs(mean_s1 - mean_s6), cmap='hot')
axes[2].set_title('Perbedaan (|Skor 2 - Skor 6|)', fontweight='bold')
axes[2].axis('off')

plt.suptitle('Heatmap Rata-rata Piksel: Skor 2 vs Skor 6', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

diff_sev = np.mean(np.abs(mean_s1 - mean_s6))
print(f'Rata-rata selisih intensitas piksel: {diff_sev:.2f} (dari skala 0-255)')
print(f'Semakin tinggi, semakin jelas perbedaan antara keparahan ringan dan parah.')

**Jawaban Pertanyaan Bisnis 2:**

> ✅ **Ya, Severity Score 6 secara visual jauh berbeda dari Severity Score 1.**

Bukti pendukung:
1. **Sampel visual** menunjukkan perbedaan mencolok — Skor 1 masih menampilkan huruf yang terbaca jelas, sementara Skor 6 hampir sepenuhnya tertutup *scribbling* destruktif.
2. **Heatmap perbedaan** menunjukkan area selisih yang tersebar luas, mengindikasikan bahwa pola piksel antara kedua skor sangat berbeda secara kuantitatif.
3. Temuan ini **mendukung hipotesis awal**: *"Tingkat keparahan goresan dapat dikuantifikasi secara bertingkat"* — model AI memiliki landasan visual yang kuat untuk membedakan derajat keparahan.

### Kesimpulan Explanatory Analysis

| Pertanyaan Bisnis | Jawaban | Bukti |
|---|---|---|
| Apakah perbedaan distribusi visual Normal vs Disleksia signifikan? | ✅ **Ya** | Sampel visual + heatmap rata-rata piksel |
| Apakah Severity Score 6 berbeda dari Score 1? | ✅ **Ya** | Sampel visual + heatmap perbedaan intensitas |

**Kedua hipotesis awal TERBUKTI didukung oleh data.**

Implikasi untuk proyek DyslexiaLens:
1. Model CNN **layak dan memungkinkan** untuk membedakan Normal vs Disleksia berdasarkan pola goresan.
2. Skala severity (0–6) memiliki **dasar visual yang kuat** — bukan sekadar label arbitrer, tetapi merepresentasikan perbedaan nyata pada tingkat piksel.
3. Sistem DyslexiaLens akan mampu memberikan output berupa **skor keparahan yang bermakna**, bukan hanya klasifikasi biner Ya/Tidak.

---
# Tahap 5: Stratified Split & Persiapan Handover

Tahap ini membagi data bersih menjadi **3 set** (Train / Validation / Test) dan menyiapkan seluruh informasi yang dibutuhkan AI Engineer.

### Pendekatan: Data Bersih Tanpa Augmentasi Fisik

Berdasarkan analisis EDA, kami memilih pendekatan **konservatif** di mana:
- Data Scientist menyiapkan **data bersih + split yang seimbang**
- Augmentasi dilakukan **secara on-the-fly** oleh AI Engineer di training pipeline (via `ImageDataGenerator` atau `tf.data.Dataset`)

**Alasan:**
1. Menghindari *distribution shift* antara Train dan Validation/Test
2. Augmentasi on-the-fly menghasilkan variasi **tak terbatas** (tidak terbatas 4 copy)
3. AI Engineer memiliki fleksibilitas penuh untuk menyesuaikan parameter augmentasi sesuai kebutuhan model
4. Mencegah potensi **data leakage** dari augmentasi offline

> 📦 **Versi alternatif** dengan augmentasi offline tersedia di `Dyslexia_OfflineAugmentation.ipynb`

### 5A. Stratified Split: Train → Train_actual + Validation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# ========================
# KONFIGURASI
# ========================
CSV_INPUT = 'master_dataset_dyslexia.csv'
CSV_OUTPUT = 'master_dataset_final.csv'
VAL_RATIO = 0.2  # 20% dari Train asli → Validation

# Load data bersih
df = pd.read_csv(CSV_INPUT)
train_orig = df[df['split'] == 'Train'].copy()
test_data = df[df['split'] == 'Test'].copy()

print(f'Data bersih: {len(df):,} baris')
print(f'Train original: {len(train_orig):,}')
print(f'Test (tidak disentuh): {len(test_data):,}')

# ========================
# STRATIFIED SPLIT
# ========================
train_actual, val_data = train_test_split(
    train_orig,
    test_size=VAL_RATIO,
    random_state=42,
    stratify=train_orig['severity_score']
)

train_actual = train_actual.copy()
val_data = val_data.copy()
val_data['split'] = 'Validation'

# Gabung & simpan
df_final = pd.concat([train_actual, val_data, test_data], ignore_index=True)
df_final.to_csv(CSV_OUTPUT, index=False)

print(f'\n=== HASIL STRATIFIED SPLIT ===')
total = len(df_final)
for split in ['Train', 'Validation', 'Test']:
    n = len(df_final[df_final['split'] == split])
    print(f'{split:12s}: {n:>8,} gambar ({n/total*100:5.1f}%)')
print(f'{"TOTAL":12s}: {total:>8,} gambar')

print(f'\n=== DISTRIBUSI SEVERITY PER SPLIT ===')
for split in ['Train', 'Validation', 'Test']:
    subset = df_final[df_final['split'] == split]
    normal = len(subset[subset['target_class'] == 0])
    dys = len(subset[subset['target_class'] == 1])
    sev = dict(subset['severity_score'].value_counts().sort_index())
    print(f'\n\U0001f4c2 {split}: Normal={normal:,} | Disleksia={dys:,} | Rasio=1:{dys/normal:.2f}')
    print(f'   Severity: {sev}')

print(f'\n\u2705 Disimpan ke: {CSV_OUTPUT}')

### 5B. Hitung Class Weights untuk AI Engineer

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

train_final = df_final[df_final['split'] == 'Train']

# Binary
binary_classes = np.array([0, 1])
binary_weights = compute_class_weight('balanced', classes=binary_classes, y=train_final['target_class'].values)
binary_weight_dict = dict(zip(binary_classes.astype(int), binary_weights))

# Severity
severity_classes = np.sort(train_final['severity_score'].unique())
severity_weights = compute_class_weight('balanced', classes=severity_classes, y=train_final['severity_score'].values)
severity_weight_dict = dict(zip(severity_classes.astype(int), severity_weights))

print('=== CLASS WEIGHTS UNTUK AI ENGINEER ===')
print(f'\n\U0001f4ca Binary (target_class):')
for cls, w in binary_weight_dict.items():
    label = 'Normal' if cls == 0 else 'Disleksia'
    print(f'   Kelas {cls} ({label}): {w:.4f}')

print(f'\n\U0001f4ca Severity (severity_score):')
for cls, w in severity_weight_dict.items():
    print(f'   Skor {cls}: {w:.4f}')

print(f'\n\U0001f4a1 Gunakan dictionary ini sebagai parameter class_weight di model.fit()')

### 5C. Validasi Visual Distribusi Split

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import os

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

fig = plt.figure(figsize=(18, 10))

# --- Panel 1: Pie chart proporsi split ---
ax1 = fig.add_subplot(2, 3, 1)
split_sizes = df_final['split'].value_counts().reindex(['Train', 'Validation', 'Test'])
colors_split = ['#3498db', '#f39c12', '#2ecc71']
ax1.pie(split_sizes, labels=[f'{s}\n({v:,})' for s, v in split_sizes.items()],
        colors=colors_split, autopct='%1.1f%%', startangle=90,
        textprops={'fontsize': 10, 'fontweight': 'bold'},
        wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax1.set_title('Proporsi Split', fontweight='bold', fontsize=12)

# --- Panel 2: Binary class per split ---
ax2 = fig.add_subplot(2, 3, 2)
binary_data = df_final.groupby(['split', 'target_class']).size().unstack(fill_value=0)
binary_data = binary_data.reindex(['Train', 'Validation', 'Test'])
binary_data.columns = ['Normal', 'Disleksia']
binary_data.plot(kind='bar', ax=ax2, color=['#2ecc71', '#e74c3c'], edgecolor='white')
ax2.set_title('Binary Class per Split', fontweight='bold', fontsize=12)
ax2.set_ylabel('Jumlah')
ax2.tick_params(axis='x', rotation=0)
ax2.legend(fontsize=9)

# --- Panel 3: Severity proporsi per split (normalized) ---
ax3 = fig.add_subplot(2, 3, 3)
for split in ['Train', 'Validation', 'Test']:
    subset = df_final[df_final['split'] == split]
    props = subset['severity_score'].value_counts().sort_index() / len(subset) * 100
    ax3.plot(props.index, props.values, marker='o', label=split, linewidth=2)
ax3.set_title('Proporsi Severity (%) per Split', fontweight='bold', fontsize=12)
ax3.set_xlabel('Severity Score')
ax3.set_ylabel('Proporsi (%)')
ax3.set_xticks(range(7))
ax3.legend()

# --- Panel 4-6: Sampel gambar acak dari tiap split ---
for idx, split in enumerate(['Train', 'Validation', 'Test']):
    ax = fig.add_subplot(2, 6, 7 + idx*2)
    subset = df_final[(df_final['split'] == split) & (df_final['target_class'] == 0)]
    sample = subset.sample(1, random_state=42).iloc[0]
    if os.path.exists(sample['image_path']):
        ax.imshow(mpimg.imread(sample['image_path']), cmap='gray')
    ax.set_title(f'{split}\nNormal', fontsize=9, fontweight='bold')
    ax.axis('off')
    
    ax_d = fig.add_subplot(2, 6, 8 + idx*2)
    subset_d = df_final[(df_final['split'] == split) & (df_final['target_class'] == 1)]
    sample_d = subset_d.sample(1, random_state=42).iloc[0]
    if os.path.exists(sample_d['image_path']):
        ax_d.imshow(mpimg.imread(sample_d['image_path']), cmap='gray')
    ax_d.set_title(f'{split}\nDisleksia', fontsize=9, fontweight='bold')
    ax_d.axis('off')

plt.suptitle('Validasi Visual Dataset Final (Tanpa Augmentasi Fisik)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Cek konsistensi proporsi
print('\n=== PROPORSI SEVERITY PER SPLIT (%) ===')
for split in ['Train', 'Validation', 'Test']:
    subset = df_final[df_final['split'] == split]
    props = (subset['severity_score'].value_counts().sort_index() / len(subset) * 100).round(1)
    print(f'{split}: {dict(props)}')
print('\n\u2705 Jika ketiga garis di line chart berhimpitan, stratified split berhasil.')

---
# Tahap 6: Kesiapan Data untuk Pemodelan

Tahap terakhir — memastikan dataset siap diserahkan ke AI Engineer.

### 6A. Data Dictionary

Definisi formal setiap kolom dalam `master_dataset_final.csv`:

| Kolom | Tipe Data | Deskripsi | Nilai |
|---|---|---|---|
| `image_path` | String | Path relatif ke file gambar `.png` | Contoh: `Gambo/Train/Normal/Normal0001.png` |
| `file_name` | String | Nama file gambar (tanpa path) | Contoh: `Normal0001.png` |
| `split` | String | Pembagian dataset | `Train`, `Validation`, `Test` |
| `folder_category` | String | Kategori kelas asal | `Normal`, `Corrected`, `Reversal` |
| `severity_score` | Integer | Skor keparahan (normalisasi) | `0` = Normal, `1`–`6` = Ringan–Parah |
| `target_class` | Integer | Label biner | `0` = Normal, `1` = Disleksia |

### Aturan untuk AI Engineer

| Aturan | Penjelasan |
|---|---|
| **Input model** | Hanya `image_path` (baca piksel gambar) |
| **Output utama** | `target_class` (klasifikasi biner) |
| **Output sekunder** | `severity_score` (multi-class) |
| **JANGAN sebagai fitur** | `severity_score`, `folder_category`, `file_name` — data leakage |
| **Format gambar** | Grayscale, 28×28 piksel, `.png` |
| **Horizontal Flip** | ❌ DILARANG — mengubah `b` menjadi `d` |

### 6B. Rekomendasi Augmentasi untuk AI Engineer

Data yang diserahkan adalah **data asli tanpa augmentasi fisik**. AI Engineer **sangat disarankan** menerapkan augmentasi **on-the-fly** saat training:

```python
# Contoh menggunakan TensorFlow/Keras:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rotation_range=10,        # Rotasi ±10°
    zoom_range=0.1,           # Scaling 0.9×–1.1×
    width_shift_range=0.05,   # Translasi horizontal kecil
    height_shift_range=0.05,  # Translasi vertikal kecil
    horizontal_flip=False,    # ❌ JANGAN — merusak Reversal
    fill_mode='nearest',
    rescale=1./255
)

# Validation & Test: HANYA rescale, tanpa augmentasi
val_test_datagen = ImageDataGenerator(rescale=1./255)
```

### Keuntungan On-the-fly vs Offline

| Aspek | On-the-fly (Rekomendasi) | Offline (Alternatif) |
|---|---|---|
| Variasi augmentasi | ∞ (setiap epoch berbeda) | Terbatas (4 copy tetap) |
| Disk space | Hemat | +103K file tambahan |
| Data leakage risk | Nol | Ada jika split salah urutan |
| Distribution shift | Nol | Train ≠ Val/Test |
| Fleksibilitas | AI Engineer bisa tune | Sudah fixed |

> 📦 Jika AI Engineer tetap ingin menggunakan augmentasi offline, file `Dyslexia_OfflineAugmentation.ipynb` tersedia sebagai alternatif.

### 6C. Validasi Akhir

In [ ]:
df_check = pd.read_csv(CSV_OUTPUT)

print('=== VALIDASI AKHIR: master_dataset_final.csv ===')
print(f'Total baris: {len(df_check):,}')
print(f'Kolom: {list(df_check.columns)}')

# Cek proporsi
total = len(df_check)
print(f'\n=== PROPORSI SPLIT ===')
for split in ['Train', 'Validation', 'Test']:
    n = len(df_check[df_check['split'] == split])
    print(f'{split:12s}: {n:>8,} gambar ({n/total*100:5.1f}%)')

# Cek data leakage
val_paths = set(df_check[df_check['split'] == 'Validation']['image_path'])
train_paths = set(df_check[df_check['split'] == 'Train']['image_path'])
test_paths = set(df_check[df_check['split'] == 'Test']['image_path'])

print(f'\n=== CEK DATA LEAKAGE ===')
print(f'Train \u2229 Validation: {len(train_paths & val_paths)}')
print(f'Train \u2229 Test      : {len(train_paths & test_paths)}')
print(f'Validation \u2229 Test : {len(val_paths & test_paths)}')

if len(train_paths & val_paths) == 0 and len(train_paths & test_paths) == 0:
    print('\n\u2705 AMAN — Nol data leakage di semua split.')
else:
    print('\n\u26a0\ufe0f PERINGATAN — Ada overlap!')

print(f'\n\u2705 Dataset siap untuk handover ke AI Engineer.')

### Kesimpulan Pipeline Data Scientist

| Tahap | Output | Status |
|---|---|---|
| 1. Assessing Data | Audit distribusi, resolusi, anomali | ✅ |
| 2. Physical Renaming | Normalisasi skala severity 1–6 | ✅ (Opsional) |
| 3. Cleaning Data | `master_dataset_dyslexia.csv` (180.726 baris) | ✅ |
| 4. EDA + Explanatory | Visualisasi distribusi, heatmap, jawaban bisnis | ✅ |
| 5. Stratified Split | Train/Validation/Test (stratified by severity) | ✅ |
| 6. Kesiapan Data | Data Dictionary + class weights + rekomendasi | ✅ |

**File output untuk AI Engineer:**

| File | Deskripsi |
|---|---|
| `master_dataset_final.csv` | Dataset bersih dengan 3 split, tanpa augmentasi fisik |
| `Dyslexia_OfflineAugmentation.ipynb` | Alternatif dengan augmentasi offline (skor 2–5 ×5) |

**Handover ke AI Engineer:**
1. Gunakan `master_dataset_final.csv` sebagai *single source of truth*
2. Terapkan augmentasi **on-the-fly** (contoh kode di sel 6B)
3. Gunakan `class_weight` dari sel 5B
4. Evaluasi dengan **F1-Score** dan **Recall**
5. **JANGAN** gunakan Horizontal Flip